<a href="https://colab.research.google.com/github/Prashant43226/CUDA-Programming/blob/main/Cuda_programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

CUDA Programming Examples from Basic

Start with initializing the environment

In [2]:
!nvidia-smi
!nvcc --version

Tue Jun 30 05:57:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Program to check the thread index and how it behaves when Blocksize is more than warp size

In [36]:
%%writefile hello.cu
#include<stdio.h>
#include<cuda.h>
__global__ void hello(){
    int idx=threadIdx.x;
    printf("Printing thread idx%d",idx);
}
int main(){
    hello<<<1,1000>>>();
    cudaDeviceSynchronize();
    return 0;
}

Overwriting hello.cu


In [37]:
!nvcc hello.cu -o hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [38]:
!./hello

Printing thread idx992Printing thread idx993Printing thread idx994Printing thread idx995Printing thread idx996Printing thread idx997Printing thread idx998Printing thread idx999Printing thread idx736Printing thread idx737Printing thread idx738Printing thread idx739Printing thread idx740Printing thread idx741Printing thread idx742Printing thread idx743Printing thread idx744Printing thread idx745Printing thread idx746Printing thread idx747Printing thread idx748Printing thread idx749Printing thread idx750Printing thread idx751Printing thread idx752Printing thread idx753Printing thread idx754Printing thread idx755Printing thread idx756Printing thread idx757Printing thread idx758Printing thread idx759Printing thread idx760Printing thread idx761Printing thread idx762Printing thread idx763Printing thread idx764Printing thread idx765Printing thread idx766Printing thread idx767Printing thread idx864Printing thread idx865Printing thread idx866Printing thread idx867Printing thread idx868Printing t

Program to print Block size and thread index and how it behaves when run in parallel

In [39]:
%%writefile blockIdx.cu
#include<stdio.h>
#include<cuda.h>
__global__ void hello(){
  printf("%d %d ",blockIdx.x,threadIdx.x);
}
int main(){
  hello<<<2,4>>>();
  cudaDeviceSynchronize();
  return 0;
}

Writing blockIdx.cu


In [40]:
!nvcc blockIdx.cu -o blockIdx

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [41]:
!./blockIdx

1 0 1 1 1 2 1 3 0 0 0 1 0 2 0 3 

Program to add 2 arrays parallely in CUDA

In [66]:
%%writefile add.cu
#include<stdio.h>
#include<cuda.h>

__global__ void add(int *A,int *B,int *C){
  int idx=blockIdx.x*blockDim.x+threadIdx.x;
  C[idx]=A[idx]+B[idx];
}
int main(){
  int N=10;
  int a[N]={1,2,3,4,5,6,7,8,9,10};
  int b[N]={11,12,13,14,15,16,17,18,19,20};
  int C[N];
  int * da,*db,*dc;

  cudaMalloc(&da,N*sizeof(int));
  cudaMalloc(&db,N*sizeof(int));
  cudaMalloc(&dc,N*sizeof(int));

  cudaMemcpy(da,a,N*sizeof(int),cudaMemcpyHostToDevice);
  cudaMemcpy(db,b,N*sizeof(int),cudaMemcpyHostToDevice);

  add<<<1,N>>>(da,db,dc);

  cudaMemcpy(C,dc,N*sizeof(int),cudaMemcpyDeviceToHost);

  for(int i=0;i<N;++i){
    printf("%d ",C[i]);
  }

  cudaFree(da);
  cudaFree(db);
  cudaFree(dc);

}

Overwriting add.cu


In [67]:
!nvcc add.cu -o add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [68]:
!./add

12 14 16 18 20 22 24 26 28 30 

Addition of 2 vectors with different blocksize and threadpool size

In [69]:
%%writefile variableSizeAdd.cu
#include<stdio.h>
#include<cuda.h>

__global__ void add(int *A,int *B,int *C){
  int idx=blockIdx.x*blockDim.x+threadIdx.x;
  C[idx]=A[idx]+B[idx];
}
int main(){
  int N=1000;
  int a[N];
  int b[N];
  int C[N];
  int * da,*db,*dc;
  int threads=256;
  int blocks=(N+threads-1)/threads;

  for(int i=0;i<1000;++i){
    a[i]=i;
  }
  for(int i=0;i<1000;++i){
    b[i]=10+i;
  }

  cudaMalloc(&da,N*sizeof(int));
  cudaMalloc(&db,N*sizeof(int));
  cudaMalloc(&dc,N*sizeof(int));

  cudaMemcpy(da,a,N*sizeof(int),cudaMemcpyHostToDevice);
  cudaMemcpy(db,b,N*sizeof(int),cudaMemcpyHostToDevice);

  add<<<blocks,threads>>>(da,db,dc);

  cudaMemcpy(C,dc,N*sizeof(int),cudaMemcpyDeviceToHost);

  for(int i=0;i<N;++i){
    printf("%d ",C[i]);
  }

  cudaFree(da);
  cudaFree(db);
  cudaFree(dc);

}

Writing variableSizeAdd.cu


In [70]:
!nvcc variableSizeAdd.cu -o variableSizeAdd

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [71]:
!./variableSizeAdd

10 12 14 16 18 20 22 24 26 28 30 32 34 36 38 40 42 44 46 48 50 52 54 56 58 60 62 64 66 68 70 72 74 76 78 80 82 84 86 88 90 92 94 96 98 100 102 104 106 108 110 112 114 116 118 120 122 124 126 128 130 132 134 136 138 140 142 144 146 148 150 152 154 156 158 160 162 164 166 168 170 172 174 176 178 180 182 184 186 188 190 192 194 196 198 200 202 204 206 208 210 212 214 216 218 220 222 224 226 228 230 232 234 236 238 240 242 244 246 248 250 252 254 256 258 260 262 264 266 268 270 272 274 276 278 280 282 284 286 288 290 292 294 296 298 300 302 304 306 308 310 312 314 316 318 320 322 324 326 328 330 332 334 336 338 340 342 344 346 348 350 352 354 356 358 360 362 364 366 368 370 372 374 376 378 380 382 384 386 388 390 392 394 396 398 400 402 404 406 408 410 412 414 416 418 420 422 424 426 428 430 432 434 436 438 440 442 444 446 448 450 452 454 456 458 460 462 464 466 468 470 472 474 476 478 480 482 484 486 488 490 492 494 496 498 500 502 504 506 508 510 512 514 516 518 520 522 524 526 528 530 5

Vector Scaling problem

In [80]:
%%writefile vectorScaling.cu
#include<stdio.h>
#include<cuda.h>

__global__ void scale(int * a,int * c,int k,int N){
  int idx=blockIdx.x*blockDim.x+threadIdx.x;
  if(idx<N) //Important to add this statement else accesses invalid memory locations though result shows up fine
  c[idx]=a[idx]*k;
}
int main(){
  int N=1000;
  int a[N];
  int C[N];
  int k=5;
  int * da,*dc;
  int threads=256;
  int blocks=(N+threads-1)/threads;

  for(int i=0;i<1000;++i){
    a[i]=i;
  }


  cudaMalloc(&da,N*sizeof(int));
  cudaMalloc(&dc,N*sizeof(int));

  cudaMemcpy(da,a,N*sizeof(int),cudaMemcpyHostToDevice);

  scale<<<blocks,threads>>>(da,dc,k,N);

  cudaMemcpy(C,dc,N*sizeof(int),cudaMemcpyDeviceToHost);

  for(int i=0;i<N;++i){
    printf("%d ",C[i]);
  }

  cudaFree(da);
  cudaFree(dc);
}

Overwriting vectorScaling.cu


In [81]:
!nvcc vectorScaling.cu -o vectorScaling

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [82]:
!./vectorScaling

0 5 10 15 20 25 30 35 40 45 50 55 60 65 70 75 80 85 90 95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175 180 185 190 195 200 205 210 215 220 225 230 235 240 245 250 255 260 265 270 275 280 285 290 295 300 305 310 315 320 325 330 335 340 345 350 355 360 365 370 375 380 385 390 395 400 405 410 415 420 425 430 435 440 445 450 455 460 465 470 475 480 485 490 495 500 505 510 515 520 525 530 535 540 545 550 555 560 565 570 575 580 585 590 595 600 605 610 615 620 625 630 635 640 645 650 655 660 665 670 675 680 685 690 695 700 705 710 715 720 725 730 735 740 745 750 755 760 765 770 775 780 785 790 795 800 805 810 815 820 825 830 835 840 845 850 855 860 865 870 875 880 885 890 895 900 905 910 915 920 925 930 935 940 945 950 955 960 965 970 975 980 985 990 995 1000 1005 1010 1015 1020 1025 1030 1035 1040 1045 1050 1055 1060 1065 1070 1075 1080 1085 1090 1095 1100 1105 1110 1115 1120 1125 1130 1135 1140 1145 1150 1155 1160 1165 1170 1175 1180 1185 1190 1195 1200 1205 1210 1215 12